Comparison with baseline decision stump $We\_Re < We\_Re_\text{crit}$

where $We\_Re_\text{crit}$ in range(140,160)

# Libraries

In [1]:
import os
from pathlib import Path
import numpy as np

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
RANDOM_STATE = 42

In [3]:
# train/test split
import sys
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [4]:
# dict with num features
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import dict_num_features

# Selected models metrics

In [5]:
df_metrics = pd.DataFrame()
path_models = Path('../results/models_modelling_2/')

In [6]:
def calc_current_metrics(target, model, fnames=False):
    path_pipeline = path_models / model
    pipeline = joblib.load(path_pipeline)
    _, test = get_train_test(
        dataset_filename='df_modelling_dimensionless',
        target=target)
    if fnames: test = test[pipeline.steps[0][1].feature_names_+[target]]
    else: test = test[dict_num_features['df_modelling_dimensionless']\
                    +['wettability']+[target]]
    preds = pipeline.predict(test.drop(columns=[target]))
    df_current_metrics = pd.DataFrame({
            'model': [model],
            'target': [target],
            'accuracy': [accuracy_score(test[target], preds)],
            'f1': [f1_score(test[target], preds)],
            'precision': [precision_score(test[target], preds)],
            'recall': [recall_score(test[target], preds)],
            'roc_auc': [roc_auc_score(test[target], preds)]})
    return df_current_metrics

../results/models_modelling_2/logisticregression_splashing_df_modelling_dimensionless_onehot

In [7]:
df_current_metrics = calc_current_metrics(target='splashing', 
                                        model='logisticregression_splashing_df_modelling_dimensionless_onehot')
df_metrics = pd.concat((df_current_metrics, df_metrics))

../results/models_modelling_2/logisticregression_net_impact_df_modelling_dimensionless_ordenc

In [8]:
df_current_metrics = calc_current_metrics(target='net_impact', 
                                        model='logisticregression_net_impact_df_modelling_dimensionless_ordenc')
df_metrics = pd.concat((df_current_metrics, df_metrics))

../results/models_modelling_2/catboostclassifier_splashing_df_modelling_dimensionless

In [9]:
df_current_metrics = calc_current_metrics(target='splashing', 
                                        model='catboostclassifier_splashing_df_modelling_dimensionless',
                                        fnames=True)
df_metrics = pd.concat((df_current_metrics, df_metrics))

../results/models_modelling_2/catboostclassifier_net_impact_df_modelling_dimensionless

In [10]:
df_current_metrics = calc_current_metrics(target='net_impact', 
                                        model='catboostclassifier_net_impact_df_modelling_dimensionless',
                                        fnames=True)
df_metrics = pd.concat((df_current_metrics, df_metrics))

In [11]:
df_metrics

,model,target,accuracy,f1,precision,recall,roc_auc
0,catboostclassifier_net_impact_df_modelling_dim...,net_impact,0.946667,0.90,0.900000,0.900000,0.931818
0,catboostclassifier_splashing_df_modelling_dime...,splashing,0.866667,0.90,0.865385,0.937500,0.839120
0,logisticregression_net_impact_df_modelling_dim...,net_impact,0.893333,0.80,0.800000,0.800000,0.863636
0,logisticregression_splashing_df_modelling_dime...,splashing,0.840000,0.88,0.846154,0.916667,0.810185


# Decision stump $We\_Re < 147$

splashing and net impact

In [12]:
for threshold in np.arange(140, 151, 0.01):

    target = 'splashing'
    _, test = get_train_test(
        dataset_filename='df_modelling_dimensionless',
        target=target)
    preds = (test['We_Re']>=threshold).astype(int)
    df_current_metrics = pd.DataFrame({
            'model': ['decision_stump'],
            'threshold': [threshold],
            'target': [target],
            'accuracy': [accuracy_score(test[target], preds)],
            'f1': [f1_score(test[target], preds)],
            'precision': [precision_score(test[target], preds)],
            'recall': [recall_score(test[target], preds)],
            'roc_auc': [roc_auc_score(test[target], preds)]})
    df_metrics = pd.concat((df_current_metrics, df_metrics))
    
    # net impact
    
    target = 'net_impact'
    _, test = get_train_test(
        dataset_filename='df_modelling_dimensionless',
        target=target)
    preds = (test['We_Re']<threshold).astype(int)
    df_current_metrics = pd.DataFrame({
            'model': ['decision_stump'],
            'threshold': [threshold],
            'target': [target],
            'accuracy': [accuracy_score(test[target], preds)],
            'f1': [f1_score(test[target], preds)],
            'precision': [precision_score(test[target], preds)],
            'recall': [recall_score(test[target], preds)],
            'roc_auc': [roc_auc_score(test[target], preds)]})
    df_metrics = pd.concat((df_current_metrics, df_metrics))

# Metrics comparison

In [13]:
df_metrics[df_metrics['target']=='splashing'].sort_values(by=['f1', 'roc_auc'], ascending=False)

,model,threshold,target,accuracy,f1,precision,recall,roc_auc
0,catboostclassifier_splashing_df_modelling_dime...,NaN,splashing,0.866667,0.900000,0.865385,0.937500,0.839120
0,logisticregression_splashing_df_modelling_dime...,NaN,splashing,0.840000,0.880000,0.846154,0.916667,0.810185
0,decision_stump,142.67,splashing,0.813333,0.860000,0.826923,0.895833,0.781250
0,decision_stump,142.66,splashing,0.813333,0.860000,0.826923,0.895833,0.781250
0,decision_stump,142.65,splashing,0.813333,0.860000,0.826923,0.895833,0.781250
...,...,...,...,...,...,...,...,...
0,decision_stump,140.59,splashing,0.773333,0.834951,0.781818,0.895833,0.725694
0,decision_stump,140.58,splashing,0.773333,0.834951,0.781818,0.895833,0.725694
0,decision_stump,140.57,splashing,0.773333,0.834951,0.781818,0.895833,0.725694
0,decision_stump,140.56,splashing,0.773333,0.834951,0.781818,0.895833,0.725694


In [14]:
df_metrics[df_metrics['target']=='net_impact'].sort_values(by=['f1', 'roc_auc'], ascending=False)

,model,threshold,target,accuracy,f1,precision,recall,roc_auc
0,catboostclassifier_net_impact_df_modelling_dim...,NaN,net_impact,0.946667,0.900000,0.900000,0.90,0.931818
0,logisticregression_net_impact_df_modelling_dim...,NaN,net_impact,0.893333,0.800000,0.800000,0.80,0.863636
0,decision_stump,149.35,net_impact,0.746667,0.641509,0.515152,0.85,0.779545
0,decision_stump,149.34,net_impact,0.746667,0.641509,0.515152,0.85,0.779545
0,decision_stump,149.33,net_impact,0.746667,0.641509,0.515152,0.85,0.779545
...,...,...,...,...,...,...,...,...
0,decision_stump,145.55,net_impact,0.706667,0.541667,0.464286,0.65,0.688636
0,decision_stump,145.54,net_impact,0.706667,0.541667,0.464286,0.65,0.688636
0,decision_stump,145.53,net_impact,0.706667,0.541667,0.464286,0.65,0.688636
0,decision_stump,145.52,net_impact,0.706667,0.541667,0.464286,0.65,0.688636
